# Synthesis and Capture

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from acadia.system import Acadia, StreamConfiguration
from acadia.channel import Channel
from acadia.arrays import ProceduralWaveform, Waveform

No module named 'pyxrfdc'
No module named 'pyxrfclk'


# System Configuration

In [2]:
acadia = Acadia()

pulse_channel = acadia.DAC(1)

def pulse_shape(out, sample_times):
    out[:] = Channel.to_samples(0.99*np.ones(len(sample_times), dtype=np.complex64))

pulse = ProceduralWaveform(pulse_channel, pulse_shape, pulse_channel)

capture_channel = acadia.ADC(1)
capture_data = Waveform(capture_channel, 5000e-9, acadia.PLDDR0Array)
capture_configuration = StreamConfiguration(capture_channel, acadia=acadia)

def configure():
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=2000e6)
    pulse_channel.set_vop(20000)
    
    capture_channel.set_nyquist_zone(2)
    capture_channel.set_dsa(0)
    

## The actual meat of the program

In [3]:
# Create a sequence for the sequencer
def sequence(a):
    with a.synchronizer():
        a.generate(pulse_channel, pulse)
        a.capture(capture_configuration, capture_data)

# Because the pulse is procedurally generated, we can change its length at runtime,
# so we need to start by allocating the length to use initially
pulse.allocate(1000e-9)

# Attach to the hardware
acadia.attach()

# Load the wave memory with the pulse by calling the generator function
pulse.populate()

# Configure channel parameters using the function we defined above
configure()

# Configure the stream processing path to capture data using the configuration
# written above
acadia.configure_stream(capture_configuration)

# Now actually run the sequencer
acadia.run(sequence)

In [ ]:
# Get the sample data from capture memory
trace = Channel.from_samples(capture_data.memory())
times = capture_data.axis()*1e6

# Plot the captured signal
fig,ax = plt.subplots()
ax.plot(times, np.real(trace), label="Re")
ax.plot(times, np.imag(trace), label="Im")
ax.set_xlabel("Time (us)")
ax.set_ylabel("Amplitude (\%FS)")
ax.grid()
ax.legend()